# SQL Fintech Analysis Project
## Анализ транзакций с оконными функциями

Этот ноутбук демонстрирует выполнение сложных SQL запросов для:
- Выявления мошенничества (Fraud Detection)
- Анализа оттока клиентов (Churn Analysis)
- Сегментации клиентов (RFM Analysis)
- Расчета метрик риска (Risk Metrics)

### Подключение к базе данных

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Настройка стиля графиков
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Путь к базе данных
DB_PATH = os.path.join('data', 'fintech_transactions.db')

# Проверка существования файла
if not os.path.exists(DB_PATH):
    print(f"❌ База данных не найдена: {DB_PATH}")
    print("Запустите сначала: python src/init_db.py")
else:
    print(f"✓ База данных подключена: {DB_PATH}")
    
# Функция для выполнения SQL запросов
def run_query(query_path):
    """Выполняет SQL запрос из файла и возвращает DataFrame"""
    conn = sqlite3.connect(DB_PATH)
    try:
        with open(query_path, 'r', encoding='utf-8') as f:
            query = f.read()
        df = pd.read_sql_query(query, conn)
        return df
    finally:
        conn.close()

## 1. Общая статистика по базе данных

In [ ]:
# Быстрая статистика
conn = sqlite3.connect(DB_PATH)

print("=== ОБЩАЯ СТАТИСТИКА ===\n")

# Количество клиентов и транзакций
query = """
SELECT 
    (SELECT COUNT(*) FROM customers) AS total_customers,
    (SELECT COUNT(*) FROM transactions) AS total_transactions,
    (SELECT SUM(amount) FROM transactions) AS total_volume,
    (SELECT AVG(amount) FROM transactions) AS avg_amount
"""
stats = pd.read_sql_query(query, conn)
print(f"Клиентов: {stats['total_customers'][0]}")
print(f"Транзакций: {stats['total_transactions'][0]}")
print(f"Общий объем: {stats['total_volume'][0]:,.2f} RUB")
print(f"Средняя транзакция: {stats['avg_amount'][0]:,.2f} RUB")

# Фрод статистика
print("\n=== ФРОД СТАТИСТИКА ===\n")
fraud_query = """
SELECT 
    COUNT(CASE WHEN is_fraud = 1 THEN 1 END) AS fraud_count,
    SUM(CASE WHEN is_fraud = 1 THEN amount ELSE 0 END) AS fraud_volume,
    COUNT(*) AS total_txns
FROM transactions
"""
fraud_stats = pd.read_sql_query(fraud_query, conn)
fraud_rate = fraud_stats['fraud_count'][0] / fraud_stats['total_txns'][0] * 100
print(f"Фрод-транзакций: {fraud_stats['fraud_count'][0]} ({fraud_rate:.2f}%)")
print(f"Объем фрода: {fraud_stats['fraud_volume'][0]:,.2f} RUB")

conn.close()

## 2. Выявление мошенничества (Fraud Detection)

Используем оконные функции для поиска аномалий:
- Z-score отклонение от среднего по клиенту
- Скользящее среднее за последние 5 транзакций
- Ранжирование по сумме

In [ ]:
# Выполняем запрос на выявление фрода
df_fraud = run_query('sql/queries/01_fraud_detection.sql')

print(f"Найдено подозрительных транзакций: {len(df_fraud)}")
print("\nТоп-10 по Z-score:")
df_fraud[['transaction_id', 'customer_id', 'amount', 'z_score', 'anomaly_flag', 'actual_fraud']].head(10)

In [ ]:
# Визуализация: распределение Z-score для фрод и легальных транзакций
plt.figure(figsize=(14, 6))

sns.boxplot(data=df_fraud, x='actual_fraud', y='z_score', palette='Set2')
plt.xlabel('Фрод (1) / Легальная (0)')
plt.ylabel('Z-score')
plt.title('Распределение Z-score для подозрительных транзакций')
plt.xticks([0, 1], ['Легальная', 'Фрод'])

plt.tight_layout()
plt.show()

# Статистика обнаружения
detected = df_fraud[df_fraud['z_score'] > 3.0]
print(f"\nТранзакций с Z-score > 3.0: {len(detected)}")
print(f"Из них реальный фрод: {detected['actual_fraud'].sum()} ({detected['actual_fraud'].sum()/len(detected)*100:.1f}%)")

## 3. Анализ оттока (Churn Analysis)

Используем LAG для расчета интервалов между транзакциями и выявления клиентов с признаками оттока.

In [ ]:
df_churn = run_query('sql/queries/02_churn_analysis.sql')

print(f"Проанализировано клиентов: {len(df_churn)}")
print("\nРаспределение по уровню риска оттока:")
print(df_churn['churn_risk_level'].value_counts())

# Топ-10 клиентов с высоким риском оттока
high_risk = df_churn[df_churn['churn_risk_level'] == 'High Churn Risk']
print(f"\nКлиенты с ВЫСОКИМ риском оттока ({len(high_risk)}):")
high_risk[['customer_name', 'segment', 'days_since_last_activity', 'max_days_gap', 'total_transactions']].head(10)

In [ ]:
# Визуализация: Days Since Last Activity по сегментам
plt.figure(figsize=(12, 6))

sns.boxplot(data=df_churn, x='churn_risk_level', y='days_since_last_activity', 
            order=['Low Risk', 'Medium Churn Risk', 'High Churn Risk'],
            palette='YlOrRd')
plt.xlabel('Уровень риска оттока')
plt.ylabel('Дней с последней активности')
plt.title('Активность клиентов по уровню риска оттока')

plt.tight_layout()
plt.show()

## 4. Сегментация клиентов (RFM Analysis)

RFM-сегментация с использованием NTILE и RANK:
- **Recency**: дней с последней покупки
- **Frequency**: количество покупок
- **Monetary**: общая сумма покупок

In [ ]:
df_rfm = run_query('sql/queries/03_customer_segmentation.sql')

print(f"Сегментировано клиентов: {len(df_rfm)}")
print("\nРаспределение по RFM-сегментам:")
print(df_rfm['customer_segment'].value_counts())

# Топ-10 клиентов (Champions)
champions = df_rfm[df_rfm['customer_segment'] == 'Champions']
print(f"\nChampions ({len(champions)} клиентов):")
champions[['customer_name', 'rfm_total_score', 'total_monetary', 'frequency']].head(10)

In [ ]:
# Визуализация: Scatter plot Frequency vs Monetary с цветом по сегменту
plt.figure(figsize=(14, 8))

segments_order = ['Champions', 'Loyal Customers', 'Regular', 'New Customers', 'At Risk', 'Lost']
palette = {'Champions': 'gold', 'Loyal Customers': 'green', 'Regular': 'blue', 
           'New Customers': 'cyan', 'At Risk': 'orange', 'Lost': 'red'}

for segment in segments_order:
    subset = df_rfm[df_rfm['customer_segment'] == segment]
    plt.scatter(subset['frequency'], subset['total_monetary'], 
               label=segment, alpha=0.7, s=80, color=palette.get(segment, 'gray'))

plt.xlabel('Frequency (количество транзакций)')
plt.ylabel('Monetary (общая сумма, RUB)')
plt.title('RFM Сегментация клиентов')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Метрики риска (Risk Metrics)

Расчет ключевых метрик:
- Fraud Rate (% от объема и количества)
- Cost of Risk
- Динамика фрода по месяцам (MoM)

In [ ]:
df_risk = run_query('sql/queries/04_risk_metrics.sql')

# Запрос возвращает несколько результатов, покажем первый (Portfolio Overview)
print("=== PORTFOLIO OVERVIEW ===")
# Первые 11 строк - это первый SELECT запроса
df_risk.head(1)

In [ ]:
# Метрики по категориям (второй SELECT в файле)
conn = sqlite3.connect(DB_PATH)

category_query = """
SELECT 
    category,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN amount ELSE 0 END), 2) AS fraud_amount,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fraud_rate_by_count
FROM transactions
GROUP BY category
ORDER BY fraud_rate_by_count DESC
"""

df_categories = pd.read_sql_query(category_query, conn)
conn.close()

print("=== ФРОД ПО КАТЕГОРИЯМ ===")
df_categories

In [ ]:
# Визуализация: Fraud Rate по категориям
plt.figure(figsize=(12, 6))

sns.barplot(data=df_categories, x='category', y='fraud_rate_by_count', palette='Reds_r')
plt.xlabel('Категория')
plt.ylabel('Fraud Rate (%)')
plt.title('Уровень фрода по категориям транзакций')
plt.xticks(rotation=45)

# Добавим значения на столбцы
for i, v in enumerate(df_categories['fraud_rate_by_count']):
    plt.text(i, v + 0.1, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Динамика фрода по месяцам (Trend Analysis)

In [ ]:
conn = sqlite3.connect(DB_PATH)

trend_query = """
SELECT 
    strftime('%Y-%m', transaction_date) AS month,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fraud_rate_pct
FROM transactions
GROUP BY strftime('%Y-%m', transaction_date)
ORDER BY month
"""

df_trend = pd.read_sql_query(trend_query, conn)
conn.close()

print("=== ДИНАМИКА ФРОДА ПО МЕСЯЦАМ ===")
df_trend

In [ ]:
# Визуализация тренда
fig, ax1 = plt.subplots(figsize=(14, 6))

# Столбцы - общее количество транзакций
bars = ax1.bar(df_trend['month'], df_trend['total_transactions'], 
               alpha=0.7, color='steelblue', label='Всего транзакций')
ax1.set_xlabel('Месяц')
ax1.set_ylabel('Количество транзакций', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xticks(range(len(df_trend)))
ax1.set_xticklabels(df_trend['month'], rotation=45)

# Линия - Fraud Rate
ax2 = ax1.twinx()
line = ax2.plot(df_trend['month'], df_trend['fraud_rate_pct'], 
                color='red', marker='o', linewidth=2, markersize=8, label='Fraud Rate (%)')
ax2.set_ylabel('Fraud Rate (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, max(df_trend['fraud_rate_pct']) * 1.5)

# Заголовок и легенда
plt.title('Динамика транзакций и уровня фрода по месяцам')
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.show()

## Итоги анализа

### Ключевые выводы:

1. **Фрод-детекция**: Оконные функции (Z-score, скользящее среднее) эффективно выявляют аномалии
2. **Отток клиентов**: LAG функция позволяет рассчитать интервалы между транзакциями без self-join
3. **RFM-сегментация**: NTILE делит клиентов на квантили для точной сегментации
4. **Метрики риска**: Fraud Rate, Cost of Risk, MoM динамика — стандартные метрики финтеха

### Технические преимущества подхода:
- Все расчеты выполняются на уровне БД (быстро для больших данных)
- Минимальный перенос данных в Python
- Легко масштабировать на реальные продакшен-базы

### Следующие шаги:
- Добавить ML-модель для классификации фрода
- Настроить автоматический ежедневный отчет через Airflow
- Создать дашборд в Power BI/Tableau